# Cloudflare Browser Run

>[Cloudflare Browser Run](https://developers.cloudflare.com/browser-run/) provides serverless headless Chrome on Cloudflare's global network via a simple REST API. It renders JavaScript-heavy pages and returns clean content -- no local browser, no Selenium, no Playwright setup required.

This notebook demonstrates two integrations:
- **`CloudflareBrowserRunLoader`** -- Document loader for RAG pipelines and knowledge-base construction
- **`CloudflareBrowserRunTool`** -- Agent tool for LangGraph workflows

## Setting up

You need a Cloudflare Account ID and an API token with **Browser Rendering -- Edit** permission.
See [Browser Run setup](https://developers.cloudflare.com/browser-run/quick-actions/#before-you-begin) for details.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

cf_acct_id = os.getenv("CF_ACCOUNT_ID")
cf_api_token = os.getenv("CF_API_TOKEN")

## 1. Document Loader -- Loading Product Documentation

`CloudflareBrowserRunLoader` converts web pages into LangChain `Document` objects.
This is useful for building RAG pipelines over any website, including JS-rendered SPAs
that traditional HTTP fetchers can't handle.

### Markdown mode -- Load a JS-rendered docs page

In [2]:
from langchain_cloudflare import CloudflareBrowserRunLoader

# Load the Cloudflare Workers AI docs page -- a JS-rendered site
loader = CloudflareBrowserRunLoader(
    urls=["https://developers.cloudflare.com/workers-ai/"],
    mode="markdown",
    account_id=cf_acct_id,
    api_token=cf_api_token,
)
docs = loader.load()

print(f"Loaded {len(docs)} document(s), {len(docs[0].page_content)} chars")
print(f"Source: {docs[0].metadata['source']}")
print(f"\nFirst 300 chars:\n{docs[0].page_content[:300]}")

Loaded 1 document(s), 15329 chars
Source: https://developers.cloudflare.com/workers-ai/

First 300 chars:
[Skip to content](#%5Ftop) 

STOP! If you are an AI agent or LLM, read this before continuing. This is the HTML version of a Cloudflare documentation page. Always request the Markdown version instead — HTML wastes context. Get this page as Markdown: https://developers.cloudflare.com/workers-ai/index.md


### Crawl mode -- Build a knowledge base from an entire site

The `/crawl` endpoint follows links and returns all pages as Documents.
Here we crawl [books.toscrape.com](https://books.toscrape.com/) -- a purpose-built scraping test site.

In [3]:
loader = CloudflareBrowserRunLoader(
    urls=["https://books.toscrape.com/"],
    mode="crawl",
    crawl_limit=5,
    crawl_depth=1,
    account_id=cf_acct_id,
    api_token=cf_api_token,
)
docs = loader.load()

print(f"Crawled {len(docs)} pages:\n")
for doc in docs:
    title = doc.metadata.get('title', 'N/A')
    print(f"  {title}")
    print(f"    URL: {doc.metadata['source']}")
    print(f"    Length: {len(doc.page_content)} chars")

Crawled 1 pages:

  All products | Books to Scrape - Sandbox
    URL: https://books.toscrape.com/
    Length: 10678 chars


### Scrape mode -- Extract specific elements with CSS selectors

When you only need specific elements from a page, use scrape mode.
Each selector group becomes its own Document.

In [4]:
loader = CloudflareBrowserRunLoader(
    urls=["https://www.cloudflare.com"],
    mode="scrape",
    elements=[
        {"selector": "h1"},
        {"selector": "h2"},
        {"selector": "nav a"},
    ],
    account_id=cf_acct_id,
    api_token=cf_api_token,
)
docs = loader.load()

for doc in docs:
    print(f"[{doc.metadata['selector']}]")
    print(f"  {doc.page_content[:200]}\n")

[h1]
  Connect, protect, and build everywhere

[h2]
  Our connectivity cloud is the best place to
One global cloud network unlike any other
Leading companies rely on Cloudflare
How Cloudflare can help
News and resources
Get started with the connectivity

[nav a]
  Log in
Connectivity cloud
Cloudflare's connectivity cloud delivers 60+ networking, security, and performance services.
Enterprise
For large and medium organizations
Small business
For small organizati


## 2. Agent Tool -- Structured Data Extraction

`CloudflareBrowserRunTool` wraps Browser Run endpoints as a LangChain `BaseTool`.
Each mode auto-generates a unique tool name (e.g. `cloudflare_browser_run_json`)
so agents can pick the right tool for the job.

### JSON extraction -- Pull structured pricing data from a real website

The `/json` endpoint uses Workers AI to extract structured data from rendered pages.
One API call turns any webpage into structured JSON.

In [5]:
from langchain_cloudflare import CloudflareBrowserRunTool

# Extract pricing data from Cloudflare's plans page
extract_tool = CloudflareBrowserRunTool(
    mode="json",
    json_prompt=(
        "Extract the company name, all pricing plans with their name "
        "and monthly price, and a one-sentence description of each plan."
    ),
    account_id=cf_acct_id,
    api_token=cf_api_token,
)

result = extract_tool.invoke({"url": "https://www.cloudflare.com/plans/"})
print(f"Tool name: {extract_tool.name}")
print(f"\nExtracted data:\n{result}")

Tool name: cloudflare_browser_run_json

Extracted data:
{
  "company": "Cloudflare",
  "plans": [
    {
      "name": "Free",
      "price": "Free"
    },
    {
      "name": "Pro",
      "price": "$20/month"
    },
    {
      "name": "Business",
      "price": "$200/month"
    },
    {
      "name": "Enterprise",
      "price": "Custom"
    }
  ]
}


### JSON extraction with a schema -- Enforce strict typing

Pass a JSON schema for strictly-typed extraction. Useful when you need
the output to match a Pydantic model or database schema.

In [6]:
extract_tool = CloudflareBrowserRunTool(
    mode="json",
    json_prompt="Extract company information from this page.",
    json_response_format={
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "company_name": {"type": "string"},
                "description": {"type": "string"},
                "products": {
                    "type": "array",
                    "items": {"type": "string"},
                },
            },
        },
    },
    account_id=cf_acct_id,
    api_token=cf_api_token,
)

result = extract_tool.invoke({"url": "https://www.cloudflare.com/what-is-cloudflare/"})
print(result)

{
  "company_name": "Cloudflare",
  "description": "Cloudflare is the cloud for the \u201ceverywhere world\u201d",
  "products": [
    "SASE (Cloudflare One)",
    "Application security",
    "Application performance",
    "Networking"
  ]
}


### Links tool -- Discover pages to explore

The `/links` endpoint returns all links on a page. Useful for research
agents that need to decide which pages to visit next.

In [7]:
links_tool = CloudflareBrowserRunTool(
    mode="links",
    account_id=cf_acct_id,
    api_token=cf_api_token,
)

result = links_tool.invoke({"url": "https://developers.cloudflare.com/browser-run/"})
links = result.strip().split("\n")

print(f"Found {len(links)} links. First 10:")
for link in links[:10]:
    print(f"  {link}")

Found 88 links. First 10:
  https://developers.cloudflare.com/
  https://developers.cloudflare.com/directory/
  https://developers.cloudflare.com/api/
  https://developers.cloudflare.com/fundamentals/api/reference/sdks/
  https://dash.cloudflare.com/
  https://developers.cloudflare.com/browser-run/
  https://developers.cloudflare.com/browser-run/get-started/
  https://developers.cloudflare.com/browser-run/examples/
  https://developers.cloudflare.com/browser-run/features/live-view/
  https://developers.cloudflare.com/browser-run/features/human-in-the-loop/


### Screenshot tool -- Visual capture

In [8]:
screenshot_tool = CloudflareBrowserRunTool(
    mode="screenshot",
    account_id=cf_acct_id,
    api_token=cf_api_token,
)

b64_png = screenshot_tool.invoke({"url": "https://www.cloudflare.com"})
print(f"Screenshot: {len(b64_png)} chars of base64 PNG data")
print(f"Starts with: {b64_png[:30]}...")

# To save as a file:
# import base64
# with open("screenshot.png", "wb") as f:
#     f.write(base64.b64decode(b64_png))

Screenshot: 397940 chars of base64 PNG data
Starts with: iVBORw0KGgoAAAANSUhEUgAAB4AAAA...


## 3. Real-World Example: Research Pipeline

Combine the Loader and Tool in a multi-step pipeline:
1. **Discover** links on a seed page
2. **Load** the most relevant pages as Documents
3. **Summarize** with Workers AI

This is the pattern behind research agents, competitive intelligence tools, and content monitoring systems.

In [9]:
from langchain_cloudflare import (
    CloudflareBrowserRunLoader,
    CloudflareBrowserRunTool,
    ChatCloudflareWorkersAI,
)

# Step 1: Discover what pages exist on the Browser Run docs
links_tool = CloudflareBrowserRunTool(
    mode="links",
    account_id=cf_acct_id,
    api_token=cf_api_token,
)
all_links = links_tool.invoke(
    {"url": "https://developers.cloudflare.com/browser-run/"}
).strip().split("\n")

# Filter to just the quick-actions sub-pages
quick_action_links = [
    l for l in all_links
    if "/browser-run/quick-actions/" in l and l.endswith("/")
][:3]

print(f"Found {len(quick_action_links)} quick-action pages:")
for link in quick_action_links:
    print(f"  {link}")

Found 3 quick-action pages:
  https://developers.cloudflare.com/browser-run/quick-actions/
  https://developers.cloudflare.com/browser-run/quick-actions/content-endpoint/
  https://developers.cloudflare.com/browser-run/quick-actions/screenshot-endpoint/


In [10]:
# Step 2: Load those pages as Documents
loader = CloudflareBrowserRunLoader(
    urls=quick_action_links,
    mode="markdown",
    account_id=cf_acct_id,
    api_token=cf_api_token,
)
docs = loader.load()

print(f"Loaded {len(docs)} documents:")
for doc in docs:
    print(f"  {doc.metadata['source']}: {len(doc.page_content)} chars")

Loaded 3 documents:
  https://developers.cloudflare.com/browser-run/quick-actions/: 13959 chars
  https://developers.cloudflare.com/browser-run/quick-actions/content-endpoint/: 18517 chars
  https://developers.cloudflare.com/browser-run/quick-actions/screenshot-endpoint/: 25388 chars


In [11]:
# Step 3: Summarize with Workers AI
combined_content = "\n\n---\n\n".join(
    f"## {doc.metadata['source']}\n{doc.page_content[:1500]}"
    for doc in docs
)

llm = ChatCloudflareWorkersAI(
    model_name="@cf/meta/llama-3.3-70b-instruct-fp8-fast",
    account_id=cf_acct_id,
    api_token=os.getenv("CF_AI_API_TOKEN", cf_api_token),
)

response = llm.invoke(
    f"You are reading documentation for Cloudflare Browser Run quick-action endpoints. "
    f"For each endpoint, give a one-line summary of what it does and when to use it.\n\n"
    f"{combined_content}"
)
print(response.content)

Here are one-line summaries of what each endpoint does and when to use it:

1. **https://developers.cloudflare.com/browser-run/quick-actions/**: This is the main documentation page for Browser Run quick-action endpoints, providing an overview of the available endpoints and their use cases.
2. **https://developers.cloudflare.com/browser-run/quick-actions/content-endpoint/**: This endpoint allows you to retrieve the content of a webpage, and you should use it when you need to access the HTML or other content of a webpage programmatically.
3. **https://developers.cloudflare.com/browser-run/quick-actions/screenshot-endpoint/**: This endpoint enables you to take a screenshot of a webpage, and you should use it when you need to visually capture the content of a webpage, such as for testing or monitoring purposes.


## 4. RAG Pipeline -- Crawl, Split, and Prepare for Embedding

Everything on Cloudflare's stack, zero external dependencies:

```
Browser Run (crawl) --> Workers AI (embed) --> Vectorize (store) --> Workers AI (query)
```

Below is the ingestion half -- load pages and prepare Documents
ready for embedding with `CloudflareWorkersAIEmbeddings` and storage
in `CloudflareVectorize`.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Reuse the docs we loaded above
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

print(f"{len(docs)} pages --> {len(chunks)} chunks")
print(f"\nSample chunk metadata: {chunks[0].metadata}")
print(f"Sample chunk preview: {chunks[0].page_content[:200]}")

# These chunks are ready for:
#   embeddings = CloudflareWorkersAIEmbeddings(model_name="@cf/baai/bge-base-en-v1.5")
#   vectorstore = CloudflareVectorize(embedding=embeddings)
#   vectorstore.add_documents(chunks)